In [1]:
import json
import csv
import re
from pathlib import Path

INPUT_FILE = "./data/fr-extract.jsonl"          # 你的输入 JSONL 文件
OUTPUT_FILE = "verlan_lexicon.csv"     # 输出 CSV 文件


def normalize_text(x):
    return x.strip() if isinstance(x, str) else ""


def category_contains_verlan(categories):
    """
    categories 可能是：
    - ["Verlan", "French nouns"]
    - [{"name": "Verlan", ...}, {"name": "...", ...}]
    """
    if not categories:
        return False

    for c in categories:
        if isinstance(c, str):
            if "verlan" in c.lower():
                return True
        elif isinstance(c, dict):
            name = str(c.get("name", ""))
            if "verlan" in name.lower():
                return True
    return False


def template_contains_verlan(etymology_templates):
    """
    etymology_templates 里有时会显式写 verlan
    """
    if not etymology_templates:
        return False

    for t in etymology_templates:
        if not isinstance(t, dict):
            continue

        name = str(t.get("name", ""))
        expansion = str(t.get("expansion", ""))

        if "verlan" in name.lower() or "verlan" in expansion.lower():
            return True

        # args 里也可能出现 verlan
        args = t.get("args", {})
        if isinstance(args, dict):
            for v in args.values():
                if "verlan" in str(v).lower():
                    return True

    return False


def text_contains_verlan(entry):
    """
    检查 etymology_text / glosses / 其他文本字段里有没有 verlan
    """
    texts = []

    # 单个 etymology_text
    if "etymology_text" in entry:
        texts.append(str(entry.get("etymology_text", "")))

    # 有些数据集也可能是 etymology_texts(list)
    if "etymology_texts" in entry:
        etys = entry.get("etymology_texts", [])
        if isinstance(etys, list):
            texts.extend([str(x) for x in etys])
        else:
            texts.append(str(etys))

    # senses 里的 glosses
    senses = entry.get("senses", [])
    if isinstance(senses, list):
        for s in senses:
            if isinstance(s, dict):
                glosses = s.get("glosses", [])
                if isinstance(glosses, list):
                    texts.extend([str(g) for g in glosses])

    for t in texts:
        if "verlan" in t.lower():
            return True

    return False


def is_verlan(entry):
    """
    判断一个词条是不是 verlan
    """
    if category_contains_verlan(entry.get("categories", [])):
        return True

    if template_contains_verlan(entry.get("etymology_templates", [])):
        return True

    if text_contains_verlan(entry):
        return True

    return False


def extract_base_form(entry):
    """
    从 etymology_text / etymology_texts 中抽原词
    例如：
    - 'Verlan de joint.'
    - 'Double verlan de arabe.'
    """
    texts = []

    if "etymology_text" in entry:
        texts.append(str(entry.get("etymology_text", "")))

    if "etymology_texts" in entry:
        etys = entry.get("etymology_texts", [])
        if isinstance(etys, list):
            texts.extend([str(x) for x in etys])
        else:
            texts.append(str(etys))

    patterns = [
        r"[Vv]erlan de\s+([^.,;:()]+)",
        r"[Dd]ouble verlan de\s+([^.,;:()]+)",
        r"[Rr]everlan(?:isation)? de\s+([^.,;:()]+)",
    ]

    for txt in texts:
        txt = normalize_text(txt)
        for pat in patterns:
            m = re.search(pat, txt)
            if m:
                return m.group(1).strip()

    return ""


def extract_gloss(entry):
    """
    抽取第一个 gloss，方便人工检查
    """
    senses = entry.get("senses", [])
    if isinstance(senses, list):
        for s in senses:
            if isinstance(s, dict):
                glosses = s.get("glosses", [])
                if isinstance(glosses, list) and glosses:
                    return " | ".join(str(g) for g in glosses)
    return ""


def extract_variants(entry):
    """
    你的样例里没有 forms，但有些条目可能会有。
    """
    forms = entry.get("forms", [])
    variants = []

    if isinstance(forms, list):
        for f in forms:
            if isinstance(f, dict):
                form = f.get("form")
                if form:
                    variants.append(str(form).strip())
            elif isinstance(f, str):
                variants.append(f.strip())

    # 去重
    seen = set()
    result = []
    for v in variants:
        if v not in seen:
            seen.add(v)
            result.append(v)

    return ";".join(result)


def main():
    rows = []

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                print(f"Skip line {line_num}: invalid JSON")
                continue

            if not is_verlan(entry):
                continue

            row = {
                "word": normalize_text(entry.get("word", "")),
                "lang": normalize_text(entry.get("lang", "")),
                "lang_code": normalize_text(entry.get("lang_code", "")),
                "pos": normalize_text(entry.get("pos", "")),
                "base_form": extract_base_form(entry),
                "variants": extract_variants(entry),
                "etymology_text": normalize_text(entry.get("etymology_text", "")),
                "gloss": extract_gloss(entry),
            }

            rows.append(row)

    # 去重：按 word + pos
    dedup = {}
    for r in rows:
        key = (r["word"], r["pos"])
        if key not in dedup:
            dedup[key] = r

    rows = list(dedup.values())
    rows.sort(key=lambda x: (x["word"], x["pos"]))

    with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "word",
                "lang",
                "lang_code",
                "pos",
                "base_form",
                "variants",
                "etymology_text",
                "gloss",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"Done. Extracted {len(rows)} verlan entries to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Done. Extracted 601 verlan entries to verlan_lexicon.csv


In [2]:
import json
import csv
import time
import requests
from pathlib import Path

# ========= 配置 =========
INPUT_JSONL = "./data/fr-extract.jsonl"                 # 你的原始 dataset
OUTPUT_TITLES_CSV = "wiktionary_verlan_titles.csv"
OUTPUT_FILTERED_JSONL = "verlan_from_category.jsonl"

API_URL = "https://fr.wiktionary.org/w/api.php"
USER_AGENT = "verlan-dataset-builder/1.0"

INCLUDE_REVERLAN = True   # 是否把 Reverlanisations 子分类也并进来


# ========= 工具函数 =========
def normalize_title(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return " ".join(s.strip().split())


def is_lemma_title(title: str) -> bool:
    """
    过滤掉非普通词条页面：
    - Annexe:
    - Modèle:
    - Catégorie:
    - Aide:
    等等
    """
    if ":" in title:
        return False
    return True


def fetch_category_members(category_name: str):
    """
    用 MediaWiki categorymembers API 拉取一个 category 下的所有页面标题
    category_name 例子:
      - "Catégorie:Verlan"
      - "Catégorie:Reverlanisations"
    """
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    titles = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "format": "json",
            "list": "categorymembers",
            "cmtitle": category_name,
            "cmlimit": "500",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        r = session.get(API_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        members = data.get("query", {}).get("categorymembers", [])
        for m in members:
            title = normalize_title(m.get("title", ""))
            if title:
                titles.append(title)

        if "continue" in data and "cmcontinue" in data["continue"]:
            cmcontinue = data["continue"]["cmcontinue"]
            time.sleep(0.2)
        else:
            break

    return titles


def save_titles_csv(titles, path):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["title"])
        for t in sorted(titles):
            writer.writerow([t])


def filter_jsonl_by_titles(input_jsonl, output_jsonl, allowed_titles):
    """
    用标题白名单筛 jsonl dataset。
    默认用 entry["word"] 匹配。
    """
    kept = 0
    total = 0

    with open(input_jsonl, "r", encoding="utf-8") as fin, \
         open(output_jsonl, "w", encoding="utf-8") as fout:

        for line_num, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            total += 1

            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue

            word = normalize_title(entry.get("word", ""))

            # 只保留法语
            if entry.get("lang_code") != "fr":
                continue

            if word in allowed_titles:
                entry["from_wiktionary_category"] = 1
                fout.write(json.dumps(entry, ensure_ascii=False) + "\n")
                kept += 1

    return total, kept



# 1) 拉 Verlan 分类
verlan_titles = fetch_category_members("Catégorie:Verlan")

# 2) 可选：拉 Reverlanisations 子分类
reverlan_titles = []
if INCLUDE_REVERLAN:
    reverlan_titles = fetch_category_members("Catégorie:Reverlanisations")

# 3) 合并并过滤非 lemma 页面
all_titles = set()
for t in verlan_titles + reverlan_titles:
    if is_lemma_title(t):
        all_titles.add(normalize_title(t))

# 4) 存标题白名单
save_titles_csv(all_titles, OUTPUT_TITLES_CSV)

# 5) 用白名单筛你的 dataset
total, kept = filter_jsonl_by_titles(INPUT_JSONL, OUTPUT_FILTERED_JSONL, all_titles)

print(f"显式标注标题数（过滤命名空间后）: {len(all_titles)}")
print(f"原始 JSONL 词条数: {total}")
print(f"匹配到的词条数: {kept}")
print(f"标题白名单已保存: {OUTPUT_TITLES_CSV}")
print(f"筛选结果已保存: {OUTPUT_FILTERED_JSONL}")



显式标注标题数（过滤命名空间后）: 312
原始 JSONL 词条数: 7336780
匹配到的词条数: 400
标题白名单已保存: wiktionary_verlan_titles.csv
筛选结果已保存: verlan_from_category.jsonl


In [3]:
import json
import csv
import re

INPUT_FILE = "verlan_from_category.jsonl"
OUTPUT_FILE = "verlan_base_forms.csv"


def norm_text(x):
    return x.strip() if isinstance(x, str) else ""


def get_etymology_texts(entry):
    texts = []

    if "etymology_text" in entry and entry["etymology_text"]:
        texts.append(str(entry["etymology_text"]))

    if "etymology_texts" in entry:
        etys = entry["etymology_texts"]
        if isinstance(etys, list):
            texts.extend(str(x) for x in etys if x)
        elif etys:
            texts.append(str(etys))

    # 有些数据会把词源写在模板展开里
    for tpl in entry.get("etymology_templates", []):
        if isinstance(tpl, dict):
            exp = tpl.get("expansion")
            if exp:
                texts.append(str(exp))

    return texts


def extract_base_form(entry):
    texts = get_etymology_texts(entry)

    patterns = [
        r"[Vv]erlan de\s+['«\"]?([^.,;:()»\"]+)",
        r"([Vv]erlan) de\s+['«\"]?([^.,;:()»\"]+)",
        r"[Dd]ouble verlan de\s+['«\"]?([^.,;:()»\"]+)",
        r"[Rr]everlan(?:isation)? de\s+['«\"]?([^.,;:()»\"]+)",
        r"[Ff]orme en verlan de\s+['«\"]?([^.,;:()»\"]+)",
        r"[Mm]ot en verlan de\s+['«\"]?([^.,;:()»\"]+)",
    ]

    for txt in texts:
        txt = norm_text(txt)
        for pat in patterns:
            m = re.search(pat, txt)
            if m:
                base = m.group(1).strip()
                base = base.strip(" '\"«»")
                return base

    return ""


def extract_gloss(entry):
    senses = entry.get("senses", [])
    if isinstance(senses, list):
        for s in senses:
            if isinstance(s, dict):
                glosses = s.get("glosses", [])
                if isinstance(glosses, list) and glosses:
                    return " | ".join(str(g) for g in glosses)
    return ""


rows = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            entry = json.loads(line)
        except json.JSONDecodeError:
            print(f"Skip invalid JSON on line {line_num}")
            continue

        # 只保留法语
        if entry.get("lang_code") != "fr":
            continue

        word = norm_text(entry.get("word", ""))
        if not word:
            continue

        base_form = extract_base_form(entry)

        rows.append({
            "verlan_form": word,
            "base_form": base_form,
            "pos": norm_text(entry.get("pos", "")),
            "etymology_text": " || ".join(get_etymology_texts(entry)),
            "gloss": extract_gloss(entry),
        })


# 去重：同一个词只保留一条
dedup = {}
for r in rows:
    key = r["verlan_form"]
    if key not in dedup:
        dedup[key] = r
    else:
        # 优先保留能抽出 base_form 的那条
        if not dedup[key]["base_form"] and r["base_form"]:
            dedup[key] = r

rows = list(dedup.values())
rows.sort(key=lambda x: x["verlan_form"].lower())

with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["verlan_form", "base_form", "pos", "etymology_text", "gloss"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Done. Saved {len(rows)} rows to {OUTPUT_FILE}")

# 顺便统计一下抽出了多少个原词
num_with_base = sum(1 for r in rows if r["base_form"])
print(f"Rows with extracted base_form: {num_with_base}")
print(f"Rows without base_form: {len(rows) - num_with_base}")

Done. Saved 311 rows to verlan_base_forms.csv
Rows with extracted base_form: 164
Rows without base_form: 147


In [4]:
input_file = "./data/fr.txt"
output_file = "meuf_sentences.txt"

with open(input_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
    for line in f_in:
        if "meuf" in line.lower():
            f_out.write(line)

print("筛选完成")

筛选完成
